In [ ]:
# ==========================================================
# Install Libraries
# ==========================================================
!pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install -U trl transformers datasets bitsandbytes accelerate -q


# ==========================================================
# Imports
# ==========================================================
import torch
import time

from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig


# ==========================================================
# Device Setup
# ==========================================================
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
else:
    device = "cpu"
    print("Using CPU")


# ==========================================================
# Load Model
# ==========================================================
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2b-it",
    max_seq_length=max_seq_length,
    dtype=torch.float16,
    load_in_4bit=False,   # fp16 full model
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model and tokenizer loaded.")


# ==========================================================
# Attach LoRA Adapters
# ==========================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)

print("LoRA adapters attached.")


# ==========================================================
# Load Dataset
# ==========================================================
dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

print(dataset)


# ==========================================================
# Format Dataset Using Gemma Chat Template
# ==========================================================
def formatting_prompts_func(examples):
    texts = []

    for instruction, context, response in zip(
        examples["instruction"],
        examples["context"],
        examples["response"]
    ):
        user_message = f"""Instruction:
{instruction}

Context:
{context}
"""

        messages = [
            {
                "role": "user",
                "content": user_message,
            },
            {
                "role": "assistant",
                "content": response,
            },
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text + tokenizer.eos_token)

    return {"text": texts}


dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    num_proc=2,
)

print("Formatted dataset ready.")


# ==========================================================
# Sanity Check: Print One Training Sample
# ==========================================================
print("\n===== SAMPLE TRAINING TEXT =====")
print(dataset[0]["text"])
print("===== END SAMPLE =====\n")


# ==========================================================
# Training Arguments
# ==========================================================
training_args = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    warmup_steps=10,
    max_steps=500,

    learning_rate=1e-4,
    fp16=True,
    bf16=False,

    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    seed=3407,
    output_dir="outputs",

    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,

    report_to="none",
)


# ==========================================================
# Build Trainer
# ==========================================================
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Trainer ready.")


# ==========================================================
# Train
# ==========================================================
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

training_start = time.time()

trainer_stats = trainer.train()

training_end = time.time()
train_time = round(training_end - training_start, 2)

train_vram_allocated = (
    round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    if device == "cuda"
    else 0.0
)

train_vram_reserved = (
    round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    if device == "cuda"
    else 0.0
)

print("\nTraining completed.")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)


# ==========================================================
# Inference Function
# ==========================================================
def run_inference(instruction, context="", max_new_tokens=256):
    FastLanguageModel.for_inference(model)

    user_message = f"""Instruction:
{instruction}

Context:
{context}
"""

    messages = [
        {
            "role": "user",
            "content": user_message,
        }
    ]

    inference_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        [inference_prompt],
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length,
    ).to(device)

    # Warm-up
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    if device == "cuda":
        torch.cuda.synchronize()

    t1 = time.time()

    input_tokens = inputs["input_ids"].shape[-1]
    generated_token_ids = output[0][input_tokens:]

    tokens_per_sec = round(len(generated_token_ids) / (t1 - t0), 2)

    answer = tokenizer.decode(
        generated_token_ids,
        skip_special_tokens=True
    )

    print("\n===== FINAL RESULTS =====")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", train_vram_allocated)
    print("Peak reserved VRAM/GB:", train_vram_reserved)
    print("Generated tokens/sec:", tokens_per_sec)
    print("Device:", device)
    print("\nMODEL ANSWER:")
    print(answer)

    return answer


# ==========================================================
# Test Inference
# ==========================================================
answer = run_inference(
    instruction="Provide a detailed explanation of the events and circumstances that led to the outbreak of World War II",
    context="Focus on the key political decisions, international tensions, and the involvement of major nations.",
    max_new_tokens=256,
)

In [ ]:
max_steps=1000
learning_rate=5e-5